# Here we will try to make a phoneme to text convertor

Prompt : 

Description of the RNN code in Markdown : 
Decoding & Inference
*   The RNN outputs **phoneme logits**.
*   These logits are passed to an **n-gram Language Model** (e.g., 3-gram or 5-gram) to decode the most likely sequence of words.
*   The decoding process involves beam search and optional rescoring.

Here are the hyperparameters for the RNN code : 
 Example instantiation based on baseline hyperparameters
model = GRUDecoder(
    neural_dim=512,
    n_units=768,
    n_days=len(data), # Assuming 'sessions' list is available from previous cells
    n_classes=41,         # 41 classes (phonemes + blank/silence)
    rnn_dropout=0.4,
    input_dropout=0.2,
    n_layers=5,
    patch_size=14,
    patch_stride=4
)

Here is the code then (from torch.nn):
        self.gru = nn.GRU(
            input_size = self.input_size,
            hidden_size = self.n_units,
            num_layers = self.n_layers,
            dropout = self.rnn_dropout, 
            batch_first = True, # The first dim of our input is the batch dim
            bidirectional = False,
        )


The phoneme list used is the following : 

    mapping:

LOGIT_TO_PHONEME = [
'BLANK',    # "BLANK" = CTC blank symbol
'AA', 'AE', 'AH', 'AO', 'AW',
'AY', 'B', 'CH', 'D', 'DH',
'EH', 'ER', 'EY', 'F', 'G',
'HH', 'IH', 'IY', 'JH', 'K',
'L', 'M', 'N', 'NG', 'OW',
'OY', 'P', 'R', 'S', 'SH',
'T', 'TH', 'UH', 'UW', 'V',
'W', 'Y', 'Z', 'ZH',
' | ',    # "|" = silence token
]



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, List

class GRUDecoder(nn.Module):
    """GRU-based decoder for neural activity to phoneme logits"""
    
    def __init__(
        self,
        neural_dim: int = 512,
        n_units: int = 768,
        n_days: int = 1,
        n_classes: int = 41,
        rnn_dropout: float = 0.4,
        input_dropout: float = 0.2,
        n_layers: int = 5,
        patch_size: int = 14,
        patch_stride: int = 4,
        bidirectional: bool = False
    ):
        super().__init__()
        
        # Store hyperparameters
        self.neural_dim = neural_dim
        self.n_units = n_units
        self.n_days = n_days
        self.n_classes = n_classes
        self.rnn_dropout = rnn_dropout
        self.input_dropout = input_dropout
        self.n_layers = n_layers
        self.patch_size = patch_size
        self.patch_stride = patch_stride
        self.bidirectional = bidirectional
        
        # Input projection with patching
        # Each patch processes multiple time steps together
        self.input_size = patch_size * neural_dim
        self.patch_proj = nn.Linear(self.input_size, n_units)
        
        # Day embedding for session adaptation
        self.day_embedding = nn.Embedding(n_days, neural_dim)
        
        # Dropout layers
        self.input_dropout_layer = nn.Dropout(input_dropout)
        
        # GRU RNN
        self.gru = nn.GRU(
            input_size=self.input_size,
            hidden_size=n_units,
            num_layers=n_layers,
            dropout=rnn_dropout if n_layers > 1 else 0,
            batch_first=True,
            bidirectional=bidirectional
        )
        
        # Output layer
        gru_output_dim = n_units * (2 if bidirectional else 1)
        self.output_layer = nn.Linear(gru_output_dim, n_classes)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights for better convergence"""
        for name, param in self.named_parameters():
            if 'weight' in name:
                if 'gru' in name:
                    # Orthogonal initialization for GRU
                    if len(param.shape) >= 2:
                        nn.init.orthogonal_(param.data)
                elif 'output_layer' in name or 'patch_proj' in name:
                    nn.init.xavier_uniform_(param.data)
            elif 'bias' in name:
                nn.init.constant_(param.data, 0)
    
    def create_patches(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert neural sequences into overlapping patches
        Args:
            x: [batch_size, seq_len, neural_dim]
        Returns:
            patches: [batch_size, new_seq_len, patch_size * neural_dim]
        """
        batch_size, seq_len, neural_dim = x.shape
        
        # Calculate new sequence length after patching
        new_seq_len = (seq_len - self.patch_size) // self.patch_stride + 1
        
        # Create patches using unfold
        # unfold works on last dimension, so we need to permute
        x = x.permute(0, 2, 1)  # [batch, neural_dim, seq_len]
        patches = x.unfold(dimension=2, size=self.patch_size, step=self.patch_stride)
        # patches shape: [batch, neural_dim, new_seq_len, patch_size]
        
        # Reshape to combine dimensions
        patches = patches.permute(0, 2, 3, 1)  # [batch, new_seq_len, patch_size, neural_dim]
        patches = patches.reshape(batch_size, new_seq_len, -1)  # Flatten last two dims
        
        return patches
    
    def forward(
        self, 
        neural_features: torch.Tensor,
        day_indices: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass of the GRU decoder
        
        Args:
            neural_features: [batch_size, seq_len, neural_dim]
            day_indices: [batch_size] integer indices for day/session embedding
        Returns:
            logits: [batch_size, new_seq_len, n_classes]
            hidden_states: [batch_size, new_seq_len, hidden_dim]
        """
        batch_size, seq_len, neural_dim = neural_features.shape
        
        # Apply day embedding if provided
        if day_indices is not None:
            day_emb = self.day_embedding(day_indices)  # [batch_size, neural_dim]
            day_emb = day_emb.unsqueeze(1)  # [batch_size, 1, neural_dim]
            neural_features = neural_features + day_emb
        
        # Create patches
        patches = self.create_patches(neural_features)  # [batch, new_seq_len, patch_size*neural_dim]
        
        # Apply input dropout
        patches = self.input_dropout_layer(patches)
        
        # Process through GRU
        gru_output, _ = self.gru(patches)  # [batch, new_seq_len, hidden_dim]
        
        # Get logits
        logits = self.output_layer(gru_output)  # [batch, new_seq_len, n_classes]
        
        return logits, gru_output


class PhonemeDecoderPipeline:
    """Complete pipeline for neural activity to text"""
    
    # Phoneme mapping from your description
    LOGIT_TO_PHONEME = [
        'BLANK',    # CTC blank symbol
        'AA', 'AE', 'AH', 'AO', 'AW',
        'AY', 'B', 'CH', 'D', 'DH',
        'EH', 'ER', 'EY', 'F', 'G',
        'HH', 'IH', 'IY', 'JH', 'K',
        'L', 'M', 'N', 'NG', 'OW',
        'OY', 'P', 'R', 'S', 'SH',
        'T', 'TH', 'UH', 'UW', 'V',
        'W', 'Y', 'Z', 'ZH',
        ' | ',    # silence token
    ]
    
    def __init__(
        self,
        neural_dim: int = 512,
        n_units: int = 768,
        n_days: int = 1,
        rnn_dropout: float = 0.4,
        input_dropout: float = 0.2,
        n_layers: int = 5,
        patch_size: int = 14,
        patch_stride: int = 4
    ):
        # Initialize GRU decoder
        self.decoder = GRUDecoder(
            neural_dim=neural_dim,
            n_units=n_units,
            n_days=n_days,
            n_classes=len(self.LOGIT_TO_PHONEME),
            rnn_dropout=rnn_dropout,
            input_dropout=input_dropout,
            n_layers=n_layers,
            patch_size=patch_size,
            patch_stride=patch_stride
        )
        
        # Language model would be loaded separately
        self.language_model = None
        self.llm_scorer = None
        
    def get_phoneme_probabilities(
        self,
        neural_data: torch.Tensor,
        day_indices: Optional[torch.Tensor] = None,
        temperature: float = 1.0
    ) -> torch.Tensor:
        """
        Convert neural data to phoneme probabilities
        
        Args:
            neural_data: [batch_size, seq_len, neural_dim]
            day_indices: optional session indices
            temperature: softmax temperature for probability calibration
        Returns:
            probabilities: [batch_size, new_seq_len, n_phonemes]
        """
        # Get logits from decoder
        logits, _ = self.decoder(neural_data, day_indices)
        
        # Apply temperature scaling
        scaled_logits = logits / temperature
        
        # Convert to probabilities
        probs = F.softmax(scaled_logits, dim=-1)
        
        return probs
    
    def decode_ctc_greedy(
        self,
        logits: torch.Tensor,
        collapse_repeated: bool = True
    ) -> List[List[str]]:
        """
        Greedy CTC decoding for phoneme sequences
        
        Args:
            logits: [batch_size, seq_len, n_classes]
            collapse_repeated: whether to collapse repeated phonemes
        Returns:
            List of phoneme sequences for each batch item
        """
        batch_size = logits.shape[0]
        predictions = []
        
        # Get most likely phoneme at each time step
        phoneme_indices = torch.argmax(logits, dim=-1)  # [batch_size, seq_len]
        
        for i in range(batch_size):
            seq = []
            prev_idx = -1
            
            for idx in phoneme_indices[i]:
                idx = idx.item()
                
                # Skip blank tokens (index 0)
                if idx == 0:
                    prev_idx = idx
                    continue
                
                # Collapse repeated phonemes if requested
                if collapse_repeated and idx == prev_idx:
                    continue
                
                # Add phoneme to sequence
                phoneme = self.LOGIT_TO_PHONEME[idx]
                seq.append(phoneme)
                prev_idx = idx
            
            predictions.append(seq)
        
        return predictions
    
    def beam_search_decoding(
        self,
        logits: torch.Tensor,
        beam_width: int = 10,
        lm_weight: float = 0.5
    ) -> List[List[str]]:
        """
        Beam search decoding with optional language model integration
        
        Args:
            logits: [batch_size, seq_len, n_classes]
            beam_width: number of beams to keep
            lm_weight: weight for language model score
        Returns:
            Best phoneme sequences for each batch item
        """
        # This is a simplified version - actual implementation would be more complex
        # with proper LM integration and CTC prefix beam search
        
        batch_size = logits.shape[0]
        all_predictions = []
        
        for b in range(batch_size):
            # Convert logits to probabilities
            probs = F.softmax(logits[b], dim=-1)
            
            # Initialize beams
            beams = [([], 0.0)]  # (sequence, score)
            
            for t in range(probs.shape[0]):
                new_beams = []
                
                for seq, score in beams:
                    # Get top-k phonemes at this time step
                    topk_probs, topk_indices = torch.topk(probs[t], beam_width)
                    
                    for i in range(beam_width):
                        phoneme_idx = topk_indices[i].item()
                        
                        # Skip blank tokens
                        if phoneme_idx == 0:
                            new_seq = seq
                        else:
                            # Add phoneme (with CTC collapse logic)
                            phoneme = self.LOGIT_TO_PHONEME[phoneme_idx]
                            if not seq or phoneme != seq[-1]:
                                new_seq = seq + [phoneme]
                            else:
                                new_seq = seq
                        
                        # Update score
                        new_score = score + torch.log(topk_probs[i]).item()
                        
                        # Add language model score if available
                        if self.language_model is not None and new_seq:
                            lm_score = self.language_model.score_sequence(new_seq)
                            new_score += lm_weight * lm_score
                        
                        new_beams.append((new_seq, new_score))
                
                # Keep only top-k beams
                new_beams.sort(key=lambda x: x[1], reverse=True)
                beams = new_beams[:beam_width]
            
            # Return best sequence
            best_seq = beams[0][0]
            all_predictions.append(best_seq)
        
        return all_predictions


# Example usage
if __name__ == "__main__":
    # Create model with your hyperparameters
    model = GRUDecoder(
        neural_dim=512,
        n_units=768,
        n_days=5,  # Example: 5 sessions
        n_classes=41,
        rnn_dropout=0.4,
        input_dropout=0.2,
        n_layers=5,
        patch_size=14,
        patch_stride=4
    )
    
    # Create complete pipeline
    pipeline = PhonemeDecoderPipeline(
        neural_dim=512,
        n_units=768,
        n_days=5,
        rnn_dropout=0.4,
        input_dropout=0.2,
        n_layers=5,
        patch_size=14,
        patch_stride=4
    )
    
    # Example forward pass
    batch_size = 4
    seq_len = 100
    neural_dim = 512
    
    # Create dummy neural data
    dummy_neural = torch.randn(batch_size, seq_len, neural_dim)
    dummy_days = torch.randint(0, 5, (batch_size,))
    
    # Get phoneme logits
    logits, hidden = model(dummy_neural, dummy_days)
    print(f"Logits shape: {logits.shape}")  # Should be [4, new_seq_len, 41]
    
    # Get phoneme probabilities
    probs = pipeline.get_phoneme_probabilities(dummy_neural, dummy_days)
    print(f"Probabilities shape: {probs.shape}")
    
    # Greedy decoding
    predictions = pipeline.decode_ctc_greedy(logits)
    print(f"Sample prediction: {predictions[0][:10]}")  # First 10 phonemes of first sequence

Logits shape: torch.Size([4, 22, 41])
Probabilities shape: torch.Size([4, 22, 41])
Sample prediction: ['AO', 'Z', 'K', 'IH', 'SH', 'AA', 'IH', 'SH', 'AA', 'SH']
